<a href="https://colab.research.google.com/github/overgroove/samsung_lecture/blob/main/04_DL_Regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Regularization**
Overfitting(과적합)이 발생하는 원인을 이해하고, 이를 억제하여 Generalization(일반화) 성능을 높이는 대표적인 정규화 기법들을 학습

## **1. DL 모델의 Overfitting**
- 모델이 학습데이터의 노이즈까지 과도하게 학습, 테스트 데이터의 성능이 떨어지는 현상
- **overfitting** 이 발생된 경우 현실 문제에 적용이 어려운 일반화 성능이 떨어진다 판단
- 딥러닝 모델의 경우 레이어가 깊어지고 unit수가 많아질 수록 모델이 표현할 수 있는 함수의 복잡도가 커짐
- 파라메터 관점에서는 모델의 파라메터 즉 weights 가 데이터에 비해 너무 많다면 최적화 과정에서 특정 파라메터가 지나치게 비대해지거나 불안정

<div align="center">
<img src="https://raw.githubusercontent.com/overgroove/ML_lecture_image_data/main/image/64.png">
</div>

## **2. 대표적인 Regularization 방법론**
- **L2 정규화 (Weight Decay)** : 손실 함수에 파라메터 제곱합을 더해 가중치 크기를 제한
- **드랍아웃 (Dropout)** : 학습 시 무작위로 뉴런을 비활성화하여 앙상블 효과 유도
- **얼리스탑핑 (Early Stopping)** : 검증 손실(Validation Loss)이 증가하는 시점에 학습을 조기 종료



### **2.2 L2 정규화 (Weight Decay)**
- 손실 함수에 파라메터 제곱합을 더해 가중치 크기를 제한
- $Loss = Original\_Loss + \frac{\lambda}{2n} \sum W^2$
- 레이어 선언 시 kernel_regularizer 옵션으로 추가
- Loss 계산이 가중치 생성과 동시에 자동으로 연동되어 계산이 빠름
- 특정 가중치가 비대해지는 문제를 해결 가능

### **2.3 드랍아웃 (Dropout)**
- 학습 시 무작위로 뉴런을 비활성화하여 앙상블 효과 유도
- 각 뉴런간의 동조화를 막고 표현력을 효과적으로 제한
- **tf.keras.layers.Dropout** 레이어를 네트워크 사이에 추가
- rate 파라메터에 비활성화 시킬 뉴런의 비율을 전달 (0.1 ~ 0.2)
- model.fit() 내부에서 학습/평가 모드에 따라 켜고 꺼짐이 자동 제어됨
- 레이어 배치 순서에 따라 성능 편차가 존재할 수 있음

## **3. overfitting 모델에 Regulazation 적용**

In [ ]:
# 모듈 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

import tensorflow as tf

In [ ]:
# load_data
data = fetch_openml('mnist_784')
y = data['target'].astype(np.int32)
X = data['data'] / 255
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 1. Load and Preprocess MNIST Data
def load_and_preprocess_data():
    # Fetch MNIST data (70,000 samples, 784 features)
    data = fetch_openml('mnist_784', version=1, as_frame=False)

    # Extract features and targets
    X = data['data'] / 255.0  # Min-Max Scaling (0 ~ 1)
    y = data['target'].astype(np.int32)

    # Split into train and validation/test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    return X_train, X_test, y_train, y_test

# 2. Overfitted Model
def build_overfitted_model():
    model = tf.keras.models.Sequential([
        tf.keras.layers.Dense(512, activation='relu', input_shape=(784,)),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax') # 10 classes for MNIST
    ])
    return model

# 3. Regularized Model (Applying L2 and Dropout and He)












# 4. Training Function
def train_mnist_model(model, X_train, y_train, epochs=30):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        verbose=1
    )
    return history

# 5. Plotting Learning Curves
def plot_learning_curves(history_overfit, history_reg):
    epochs_overfit = range(1, len(history_overfit.history['loss']) + 1)
    epochs_reg = range(1, len(history_reg.history['loss']) + 1)

    plt.figure(figsize=(14, 5))

    # Plot Loss Curves
    plt.subplot(1, 2, 1)
    plt.plot(epochs_overfit, history_overfit.history['loss'], 'r--', label='Overfit Train Loss')
    plt.plot(epochs_overfit, history_overfit.history['val_loss'], 'r-', label='Overfit Val Loss')
    plt.plot(epochs_reg, history_reg.history['loss'], 'b--', label='Reg Train Loss')
    plt.plot(epochs_reg, history_reg.history['val_loss'], 'b-', label='Reg Val Loss')
    plt.title('Loss Curves Comparison')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy Curves
    plt.subplot(1, 2, 2)
    plt.plot(epochs_overfit, history_overfit.history['accuracy'], 'r--', label='Overfit Train Acc')
    plt.plot(epochs_overfit, history_overfit.history['val_accuracy'], 'r-', label='Overfit Val Acc')
    plt.plot(epochs_reg, history_reg.history['accuracy'], 'b--', label='Reg Train Acc')
    plt.plot(epochs_reg, history_reg.history['val_accuracy'], 'b-', label='Reg Val Acc')
    plt.title('Accuracy Curves Comparison')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# 6. Main Pipeline Execution
# 1. Setting data
X_train, X_test, y_train, y_test = load_and_preprocess_data()

# 2. Train Overfitted Model (No regularization, fixed epochs)
print("\n========== Training Overfitted Model ==========")
model_overfit = build_overfitted_model()
history_overfit = train_mnist_model(
    model_overfit, X_train, y_train, epochs=30
)

# 3. Train Regularized Model (L2 + Dropout + Early Stopping)
print("\n========== Training Regularized Model ==========")
model_reg = build_regularized_model(dropout_rate=0.4, l2_lambda=1e-4)
history_reg = train_mnist_model(
    model_reg, X_train, y_train, epochs=30
)

# 4. Evaluate both models on the unseen Test Set
print("\n========== Final Evaluation on Test Set ==========")
loss_ovf, acc_ovf = model_overfit.evaluate(X_test, y_test, verbose=0)
loss_reg, acc_reg = model_reg.evaluate(X_test, y_test, verbose=0)
print(f"Overfitted Model -> Test Loss: {loss_ovf:.4f}, Test Accuracy: {acc_ovf:.4f}")
print(f"Regularized Model -> Test Loss: {loss_reg:.4f}, Test Accuracy: {acc_reg:.4f}")

# 5. Visualize the results
plot_learning_curves(history_overfit, history_reg)

## **4. Batch size**
- **Batch size(배치 크기)** 는 모델 파라메터를 한 번 업데이트할 때 사용하는 샘플 데이터의 갯수
- 전체 데이터를 한번에 처리하지 않고 나누어 처리하여 GPU 메모리 효율을 높임
- 적절히 작은 배치를 사용한다면 매 스텝마다 가중치 업데이트 방향에 약간의 무작위 노이즈가 발생
- 모델이 얕은 로컬 미니마나 안장점(Saddle Point)에 갇히지 않고 탈출하여 더 나은 전역 최적점(Global Minima)을 찾을 수 있음
- model.fit() 과정에서 **batch_size**로 설정, 32/64/128/256

In [ ]:
# batch_size 적용 된 학습 함수
def train_mnist_model(model, X_train, y_train, epochs=30, batch_size=32):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )
    return history

## **5. Batch Normalization**
- 학습 과정에서 각 레이어의 활성화 값(Activation Value) 분포를 평균 0, 분산 1로 정규화
- $\rightarrow Z = wX + b$
- Batch Normalization 적용
- $\rightarrow A = \text{Sigmoid}(Z)$
- 미니배치 단위로 입력 분포를 안정화하므로, 학습률(Learning Rate)을 크게 설정해도 발산하지 않고 빠르게 수렴
- 초기 가중치 설정이 완벽하지 않더라도 레이어를 거치며 데이터 분포가 무너지지 않기 때문에 초기화에 덜 민감
- 미니배치마다 계산되는 평균과 분산의 미세한 노이즈가 모델에 드랍아웃과 유사한 규제 효과를 주어 과적합을 방지
- 각 레이어 뒤 **layers.BatchNormalization()** 추가 (relu 사용 시 실무 권장 방식)

In [ ]:
# 원 논문 기준 적용 방식
tf.kears.models.Sequential([
    tf.keras.layers.Input(units=128, input_shape=(784,)),
    # 2. Batch Normalization
    tf.keras.layers.BatchNormalization(),
    # 3. Activation function
    tf.keras.layers.Activation('relu'),
    layers.Dense(10, activation='softmax')
])

In [ ]:
# 실무 권장 방식
tf.keras.models.Sequential([
    tf.keras.layers.Input(units=128, activation='relu', input_shape=(784,)),
    # 2. Batch Normalization applied after Activation
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(10, activation='softmax')
])

## **6. Callbacks**
- 모델의 학습 과정 중 여러가지 작업을 Callback 함수 형태로 전달가능
- 학습 효율 및 성능을 위한 방법 중 early stopping / LR scheduler 활용

### **6.1 얼리스탑핑 (Early Stopping)**
- Validation Loss가 증가하는 시점에 학습을 조기 종료
- 불필요한 연산을 줄이고 가장 일반화된 시점의 모델을 저장
- 최고의 가중치를 자동 복원(restore_best_weights) 가능
- 언제 멈춰야 할지 기준(Patience)을 설정하는 탐색이 필요함
- **tf.keras.callbacks.EarlyStopping** 객체를 학습 시 전달


### **6.2 LR 스케줄러(Learning Rate Scheduler, 학습률 스케줄러)**
- 딥러닝 모델을 학습시킬 때 시간(Epoch 또는 Step)이 흐름에 따라 학습률(Learning Rate)을 동적으로 조절하는 기능
- 학습률($\alpha$)은 모델이 정답을 찾아가는 "걸음의 크기"를 의미
- 학습 초반 학습률을 크게 설정하여 정답에 빠르게 접근
- 학습 후반 최적점에 가까워지면 학습률을 작게 설정하여 안전하게 최적점에 도달
- **tf.keras.callbacks.ReduceLROnPlateau** 객체를 학습 시 전달

In [ ]:
callback_list = []

# 학습 스케쥴러 callback 함수
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,       # New LR = Current LR * 0.2
    patience=5,       # 5 에포크 진행 됨에도 val_loss 개선이 없으면 수행
    min_lr=1e-5,      # 0.0001 보다 작게만 만들지 말것
    verbose=1
)
callback_list.append(reduce_lr)

# 얼리스탑핑 callback 함수
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=12,      # 12에포크 진행 됨에도 val_loss 개선이 없으면 수행
    restore_best_weights=True, # best_weight 롤백
    verbose=1
)
callback_list.append(early_stop)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callback_list,
    verbose=1
)